# KG1 V204 - Auto Public Adapter Triage

Executa a proxima etapa sem operacao manual intermediaria:

1. Confirma V194 como baseline.
2. Descobre adapters publicos HF compativeis com `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16`.
3. Baixa um adapter por vez.
4. Avalia primeiro `official360`.
5. Avalia `all720` somente se `official360` vencer V194.
6. Gera decisao final.
7. Se nenhum adapter passar, tenta preparar V205 e para somente se precisar de acao humana: token HF/aceite de termos para dataset gated.

Este notebook nao submete no Kaggle.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import time
import urllib.request
from datetime import datetime, timezone

def utc_now():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    print('Wrote:', path)

def run_cmd(cmd, cwd=None, env=None, check=False):
    cmd = [str(x) for x in cmd]
    print('+', ' '.join(cmd))
    p = subprocess.run(cmd, cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout[-6000:])
    if check and p.returncode != 0:
        raise RuntimeError(f'command failed rc={p.returncode}')
    return {'returncode': p.returncode, 'stdout': p.stdout}

IS_COLAB = Path('/content').exists()
WORK_DIR = Path('/content/kg1_v204_auto') if IS_COLAB else Path.cwd() / '.kg1_v204_auto'
SCRIPT_DIR = WORK_DIR / 'scripts'
OUT_DIR = Path(os.environ.get('KG1_V204_OUT', '/content/drive/MyDrive/KG1_NVIDIA_V204/auto_public_adapter_triage' if IS_COLAB else str(WORK_DIR / 'out')))
ADAPTER_ROOT = OUT_DIR / 'hf_adapters'
for p in [SCRIPT_DIR, OUT_DIR, ADAPTER_ROOT, OUT_DIR / 'eval_reports']:
    p.mkdir(parents=True, exist_ok=True)

V194_ADAPTER = Path('/content/drive/MyDrive/KG1_NVIDIA_V202D/init_adapter_v194_rank19_build/adapter')
V194_ADAPTER_SHA256 = '01259fef943bc16c31d8f7907be076cc987381a6a1bbe732b1b33c2d9f2ea95f'
V194_ALL720 = 0.16335251388243502
V194_OFFICIAL360 = 0.12171823230261604

ARCHIVE_ZIP = Path('/content/kg1_v202d/data/tonghuikang-0-87-nemotron-dataset.zip')
ARCHIVE_SHA256 = '461776d6bc44d482988d23c4e584128b66a93d2500fe7c428f4e895ab42e9eb8'
MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
MODEL_REVISION = 'cbd3fa9f933d55ef16a84236559f4ee2a0526848'

MAX_CANDIDATES = int(os.environ.get('KG1_V204_MAX_CANDIDATES', '8'))
STOP_AFTER_FIRST_PROMOTABLE = os.environ.get('KG1_V204_STOP_AFTER_FIRST_PROMOTABLE', '0') == '1'

print('WORK_DIR:', WORK_DIR)
print('OUT_DIR:', OUT_DIR)
print('MAX_CANDIDATES:', MAX_CANDIDATES)
print('NO KAGGLE SUBMIT IN THIS NOTEBOOK')


In [ ]:
if IS_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive mount skipped/failed:', repr(exc))

if not V194_ADAPTER.exists():
    raise RuntimeError(f'HUMAN ACTION REQUIRED: V194 adapter not found: {V194_ADAPTER}')
adapter_model = V194_ADAPTER / 'adapter_model.safetensors'
if not adapter_model.exists():
    raise RuntimeError(f'HUMAN ACTION REQUIRED: missing adapter_model.safetensors under {V194_ADAPTER}')
got = sha256_file(adapter_model)
print('V194 adapter_model sha256:', got)
if got != V194_ADAPTER_SHA256:
    raise RuntimeError('HUMAN ACTION REQUIRED: V194 adapter sha256 mismatch. Stop before evaluating candidates.')

if not ARCHIVE_ZIP.exists():
    print('Archive zip missing, expected:', ARCHIVE_ZIP)
    print('If this is a fresh runtime, run the V203 setup or copy /content/kg1_v202d/data/tonghuikang-0-87-nemotron-dataset.zip first.')
    raise RuntimeError('HUMAN ACTION REQUIRED: missing Tong pretokenized archive.')

run_cmd(['nvidia-smi'])


In [ ]:
RAW_SOURCES = [
    ('FELIPEACASTRO/KG1-NVIDIA', 'claude/competent-shamir', 'scripts'),
    ('FELIPEACASTRO/KG1', 'claude/competent-shamir', 'scripts'),
    ('FELIPEACASTRO/KG1-NVIDIA', 'master', 'competent-shamir/scripts'),
    ('FELIPEACASTRO/KG1', 'master', 'scripts'),
]

def download_script(name, required=True):
    dest = SCRIPT_DIR / name
    last_err = None
    for repo, ref, prefix in RAW_SOURCES:
        url = f'https://raw.githubusercontent.com/{repo}/{ref}/{prefix}/{name}'
        print('Downloading:', url)
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                data = r.read()
            dest.write_bytes(data)
            print('OK:', dest, 'bytes=', len(data))
            return dest
        except Exception as exc:
            last_err = exc
    if required:
        raise RuntimeError(f'Failed to download {name}: {last_err}')
    print('WARN optional script failed:', name, repr(last_err))
    return None

TRAIN_SCRIPT = download_script('hf_job_train_v90.py')
EVAL_SCRIPT = download_script('kg1_v202_pretokenized_adapter_eval.py')
download_script('kg1_submission_gate.py', required=False)
download_script('nemotron_submission_preflight.py', required=False)


In [ ]:
try:
    from huggingface_hub import HfApi, snapshot_download
except Exception:
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
    from huggingface_hub import HfApi, snapshot_download

api = HfApi(token=os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN'))

MANUAL_CANDIDATES = [
    'etencore/nemotron-30b-reasoning-lora',
    'Aitherium/Nemotron-3-Nano-30B-LoRA-Reasoning-v2',
    'gfinin/nemotron-reasoning-lora',
    'U2DIA/nemotron-v14-epoch1',
]
SEARCH_URLS = [
    'https://huggingface.co/api/models?other=base_model:adapter:nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16&limit=100',
    'https://huggingface.co/api/models?search=nemotron%2030b%20lora%20reasoning&limit=100',
    'https://huggingface.co/api/models?search=NVIDIA-Nemotron-3-Nano-30B%20LoRA&limit=100',
]

def http_json(url):
    with urllib.request.urlopen(url, timeout=90) as r:
        return json.loads(r.read().decode('utf-8'))

seen = []
for repo_id in MANUAL_CANDIDATES:
    if repo_id not in seen:
        seen.append(repo_id)
for url in SEARCH_URLS:
    try:
        data = http_json(url)
    except Exception as exc:
        print('Search failed:', url, repr(exc))
        continue
    for item in data:
        repo_id = item.get('id') or item.get('modelId')
        if not repo_id or repo_id in seen:
            continue
        blob = repo_id.lower() + ' ' + ' '.join(map(str, item.get('tags') or [])).lower()
        if 'nemotron' in blob and ('lora' in blob or 'adapter' in blob):
            seen.append(repo_id)

REJECT_NAME_PARTS = ['dummy', 'privacy-filter', 'gguf', 'omni-30b-a3b-reasoning-bf16', 'omni-30b-a3b-reasoning-fp8', 'nvfp4']

def inspect_repo(repo_id):
    row = {'repo_id': repo_id, 'ok': False, 'reject_reason': None, 'files': []}
    if any(part in repo_id.lower() for part in REJECT_NAME_PARTS):
        row['reject_reason'] = 'name_filter'
        return row
    try:
        info = api.model_info(repo_id, files_metadata=True)
        row['downloads'] = getattr(info, 'downloads', None)
        row['likes'] = getattr(info, 'likes', None)
        row['tags'] = list(getattr(info, 'tags', []) or [])
        siblings = getattr(info, 'siblings', []) or []
        for s in siblings:
            name = getattr(s, 'rfilename', None)
            size = getattr(s, 'size', None)
            if name:
                row['files'].append({'name': name, 'size': size})
        names = {f['name'] for f in row['files']}
        if 'adapter_config.json' not in names:
            row['reject_reason'] = 'missing_adapter_config'
            return row
        if 'adapter_model.safetensors' not in names:
            row['reject_reason'] = 'missing_adapter_model_safetensors'
            return row
        size = next((f.get('size') for f in row['files'] if f['name'] == 'adapter_model.safetensors'), None)
        row['adapter_model_size'] = size
        if size and size > 7_000_000_000:
            row['reject_reason'] = 'adapter_model_too_large'
            return row
        row['ok'] = True
        return row
    except Exception as exc:
        row['reject_reason'] = 'model_info_error'
        row['error'] = repr(exc)
        return row

inspected = [inspect_repo(repo_id) for repo_id in seen]
accepted = [r for r in inspected if r.get('ok')]
accepted.sort(key=lambda r: (-(r.get('downloads') or 0), r['repo_id'].lower()))
queue = accepted[:MAX_CANDIDATES]

payload = {'generated_at': utc_now(), 'searched': seen, 'inspected': inspected, 'queue': queue}
write_json(OUT_DIR / 'V204_public_adapter_candidate_queue.json', payload)
print('Accepted candidates:', [q['repo_id'] for q in queue])
if not queue:
    raise RuntimeError('No public HF adapter candidate passed metadata filters. Proceed to V205 gated dataset.')


In [ ]:
SPLIT_OFFICIAL360 = [
    {'name': 'official360', 'val_examples': 360, 'eval_max_examples': 360, 'exclude_categories': 'matching,concatenation,splitting,spelling,lstrip'},
]
SPLIT_BOTH = [
    {'name': 'all720', 'val_examples': 720, 'eval_max_examples': 720, 'exclude_categories': ''},
    {'name': 'official360', 'val_examples': 360, 'eval_max_examples': 360, 'exclude_categories': 'matching,concatenation,splitting,spelling,lstrip'},
]

def safe_label(repo_id):
    return repo_id.replace('/', '__').replace('-', '_').replace('.', '_')

def download_adapter(repo_id):
    local_dir = ADAPTER_ROOT / safe_label(repo_id)
    if (local_dir / 'adapter_config.json').exists() and (local_dir / 'adapter_model.safetensors').exists():
        print('Adapter already downloaded:', local_dir)
        return local_dir
    print('Downloading adapter:', repo_id)
    snapshot_download(
        repo_id=repo_id,
        local_dir=str(local_dir),
        token=os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN'),
        allow_patterns=['adapter_config.json', 'adapter_model.safetensors', '*.json', '*.safetensors'],
        ignore_patterns=['*.bin', '*.pt', '*.pth', '*.gguf', '*.md', 'README*'],
    )
    if not (local_dir / 'adapter_config.json').exists():
        raise RuntimeError(f'Downloaded repo missing adapter_config.json: {repo_id}')
    if not (local_dir / 'adapter_model.safetensors').exists():
        raise RuntimeError(f'Downloaded repo missing adapter_model.safetensors: {repo_id}')
    return local_dir

def eval_adapter(label, adapter_dir, split_specs):
    report = OUT_DIR / 'eval_reports' / f'{label}.json'
    cmd = [
        sys.executable, '-u', str(EVAL_SCRIPT),
        '--training-script', str(TRAIN_SCRIPT),
        '--adapter-dir', str(adapter_dir),
        '--archive-zip', str(ARCHIVE_ZIP),
        '--expected-archive-sha256', ARCHIVE_SHA256,
        '--output-json', str(report),
        '--label', label,
        '--split-specs-json', json.dumps(split_specs),
        '--model-name', MODEL_NAME,
        '--model-revision', MODEL_REVISION,
        '--model-device-map', 'auto',
        '--max-length', '8192',
        '--seed', '202',
    ]
    result = run_cmd(cmd)
    payload = {'label': label, 'adapter_dir': str(adapter_dir), 'returncode': result['returncode'], 'report': str(report)}
    if report.exists():
        payload['eval'] = json.loads(report.read_text(encoding='utf-8'))
    return payload

def split_loss(payload, name):
    for split in payload.get('eval', {}).get('splits', []):
        if split.get('name') == name:
            return split.get('overall_loss')
    return None

def compare_payload(payload):
    all720 = split_loss(payload, 'all720')
    official360 = split_loss(payload, 'official360')
    return {
        'all720': all720,
        'official360': official360,
        'all720_delta': None if all720 is None else all720 - V194_ALL720,
        'official360_delta': None if official360 is None else official360 - V194_OFFICIAL360,
        'passed_all720': all720 is not None and all720 < V194_ALL720,
        'passed_official360': official360 is not None and official360 < V194_OFFICIAL360,
    }

queue_payload = json.loads((OUT_DIR / 'V204_public_adapter_candidate_queue.json').read_text(encoding='utf-8'))
results_path = OUT_DIR / 'V204_public_adapter_triage_results.json'
results = []
if results_path.exists():
    results = json.loads(results_path.read_text(encoding='utf-8'))

done_repos = {r.get('repo_id') for r in results if r.get('stage') == 'final'}

for row in queue_payload['queue']:
    repo_id = row['repo_id']
    if repo_id in done_repos:
        print('Already done:', repo_id)
        continue
    started = time.time()
    record = {'repo_id': repo_id, 'started_at': utc_now(), 'stage': 'final', 'metadata': row}
    try:
        local_dir = download_adapter(repo_id)
        record['adapter_dir'] = str(local_dir)
        record['adapter_model_sha256'] = sha256_file(local_dir / 'adapter_model.safetensors')
        official_payload = eval_adapter(safe_label(repo_id) + '_official360', local_dir, SPLIT_OFFICIAL360)
        record['official360_payload'] = official_payload
        record['official360_comparison'] = compare_payload(official_payload)
        if official_payload['returncode'] == 0 and record['official360_comparison']['passed_official360']:
            both_payload = eval_adapter(safe_label(repo_id) + '_both', local_dir, SPLIT_BOTH)
            record['both_payload'] = both_payload
            record['both_comparison'] = compare_payload(both_payload)
            record['promotable'] = bool(record['both_comparison']['passed_official360'] and record['both_comparison']['passed_all720'])
        else:
            record['promotable'] = False
            record['reject_reason'] = 'official360_not_better_than_v194_or_eval_failed'
    except Exception as exc:
        record['promotable'] = False
        record['error'] = repr(exc)
    record['elapsed_sec'] = round(time.time() - started, 2)
    record['finished_at'] = utc_now()
    results.append(record)
    write_json(results_path, results)
    print('TRIAGE_RECORD:', json.dumps(record, indent=2)[:4000])
    if record.get('promotable') and STOP_AFTER_FIRST_PROMOTABLE:
        break


In [ ]:
results = json.loads((OUT_DIR / 'V204_public_adapter_triage_results.json').read_text(encoding='utf-8')) if (OUT_DIR / 'V204_public_adapter_triage_results.json').exists() else []
promotable = [r for r in results if r.get('promotable')]

decision = {
    'generated_at': utc_now(),
    'baseline': {'label': 'V194', 'all720': V194_ALL720, 'official360': V194_OFFICIAL360, 'adapter_model_sha256': V194_ADAPTER_SHA256},
    'num_candidates_evaluated': len(results),
    'promotable_count': len(promotable),
    'submit_candidate': False,
    'kaggle_submit_executed': False,
    'results_path': str(OUT_DIR / 'V204_public_adapter_triage_results.json'),
}
if promotable:
    best = sorted(promotable, key=lambda r: (r.get('both_comparison', {}).get('official360') or 999, r.get('both_comparison', {}).get('all720') or 999))[0]
    decision.update({
        'decision': 'PUBLIC_ADAPTER_PROMOTABLE_FOR_PREFLIGHT_ONLY',
        'selected_repo_id': best['repo_id'],
        'selected_adapter_dir': best.get('adapter_dir'),
        'selected_comparison': best.get('both_comparison'),
        'next_action': 'Run packaging/preflight cell in a separate no-submit notebook, then require explicit human approval before Kaggle submit.',
    })
else:
    decision.update({
        'decision': 'NO_PUBLIC_ADAPTER_BEAT_V194',
        'next_action': 'Proceed to V205 verified trace mining from gated HF dataset if token and terms are available.',
    })

write_json(OUT_DIR / 'V204_FINAL_DECISION.json', decision)
print(json.dumps(decision, indent=2))


In [ ]:
decision = json.loads((OUT_DIR / 'V204_FINAL_DECISION.json').read_text(encoding='utf-8'))
v205 = {'generated_at': utc_now(), 'stage': 'V205_gated_dataset_probe', 'dataset': 'andy279/nemotron-reasoning-challenge-raw-traces'}
if decision.get('promotable_count', 0) > 0:
    v205['status'] = 'skipped_because_adapter_promotable'
else:
    hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
    if not hf_token:
        v205.update({
            'status': 'blocked_human_action_required',
            'reason': 'HF_TOKEN/HUGGINGFACE_HUB_TOKEN missing. Accept HF dataset terms and set token before V205 mining.',
            'next_human_action': 'Open https://huggingface.co/datasets/andy279/nemotron-reasoning-challenge-raw-traces, accept terms, create HF token, set HF_TOKEN in Colab, rerun this notebook from V205 probe or run V205 mining notebook.',
        })
    else:
        try:
            try:
                import datasets  # noqa: F401
            except Exception:
                run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'datasets'], check=True)
            from datasets import load_dataset
            ds = load_dataset('andy279/nemotron-reasoning-challenge-raw-traces', split='train', streaming=True, token=hf_token)
            sample = []
            for i, row in enumerate(ds):
                sample.append({k: row.get(k) for k in list(row.keys())[:20]})
                if i >= 4:
                    break
            sample_path = OUT_DIR / 'V205_raw_traces_probe_sample.json'
            write_json(sample_path, sample)
            v205.update({'status': 'ready_for_v205_mining', 'sample_path': str(sample_path), 'next_action': 'Build solver-verified filter for cryptarithm_guess, cryptarithm_deduce, cipher.'})
        except Exception as exc:
            v205.update({
                'status': 'blocked_human_action_required',
                'reason': repr(exc),
                'next_human_action': 'Confirm HF token has access and dataset terms were accepted for andy279 raw traces.',
            })

write_json(OUT_DIR / 'V205_GATED_DATASET_PROBE.json', v205)
print(json.dumps(v205, indent=2))
if v205.get('status') == 'blocked_human_action_required':
    raise RuntimeError('HUMAN ACTION REQUIRED: ' + v205.get('reason', 'blocked'))
